In [ ]:
!pip install qai-hub

In [ ]:
!pip install qai_hub_models

In [ ]:
!qai-hub configure --api_token (Your API Key)

In [ ]:
!qai-hub list-devices

In [ ]:
!pip3 install "qai-hub[torch]"

In [ ]:
import numpy as np
import requests
import torch
from PIL import Image
from torchvision.models import mobilenet_v2

import qai_hub as hub


In [ ]:
# Using pre-trained MobileNet
torch_model = mobilenet_v2(pretrained=True)
torch_model.eval()

In [ ]:
input_shape = (1, 3, 224, 224)
example_input = torch.rand(input_shape)
traced_torch_model = torch.jit.trace(torch_model, example_input)

# Step 2: Compile model

In [ ]:
!qai-hub list-devices

In [ ]:
# Step 2: Compile model
compile_job = hub.submit_compile_job(
    model=traced_torch_model,
    device=hub.Device("Samsung Galaxy S24 (Family)"),
    input_specs=dict(image=input_shape),
    options="--target_runtime tflite",
)

*   compile_options="--target_runtime tflite"                  # Uses TensorFlow Lite
*   compile_options="--target_runtime onnx"                    # Uses ONNX runtime
*   compile_options="--target_runtime qnn_lib_aarch64_android" # Runs with Qualcomm AI Engine

# Step 3: Profile model

In [ ]:
# Step 3: Profile on cloud-hosted device
target_model = compile_job.get_target_model()

profile_job = hub.submit_profile_job(
    model=target_model,
    device=hub.Device("Samsung Galaxy S24 (Family)"),
)


In [ ]:
# Print summary
from qai_hub_models.utils.printing import print_profile_metrics_from_job
profile_data = profile_job.download_profile()
print_profile_metrics_from_job(profile_job, profile_data)

*   profile_options="--compute_unit cpu"     # Use cpu
*   profile_options="--compute_unit gpu"     # Use gpu (with cpu fallback)
*   profile_options="--compute_unit npu"     # Use npu (with cpu fallback)

### Runs a performance profile on-device
```
profile_job_expt = qai_hub.submit_profile_job(
    model=target_model,                     # Compiled model
    device=device,                          # Device
    options=profile_options,
)

```



In [ ]:
profile_job_expt = hub.submit_profile_job(
    model=target_model,                     # Compiled model
    device=hub.Device("Samsung Galaxy S24 (Family)"),                       # Device
    options="--compute_unit cpu",
)

In [ ]:
profile_data = profile_job_expt.download_profile()
print_profile_metrics_from_job(profile_job_expt, profile_data)

# Step 4: Run inference on cloud-hosted device

In [ ]:
# Step 4: Run inference on cloud-hosted device
# https://qaihub-public-assets.s3.us-west-2.amazonaws.com/qai-hub-models/models/imagenet_classifier/v1/dog.jpg
# https://qaihub-public-assets.s3.us-west-2.amazonaws.com/apidoc/input_image1.jpg
sample_image_url = (
    "https://qaihub-public-assets.s3.us-west-2.amazonaws.com/apidoc/input_image1.jpg"
)
response = requests.get(sample_image_url, stream=True)
response.raw.decode_content = True
image = Image.open(response.raw).resize((224, 224))
input_array = np.expand_dims(
    np.transpose(np.array(image, dtype=np.float32) / 255.0, (2, 0, 1)), axis=0
)

In [ ]:
inference_job = hub.submit_inference_job(
    model=target_model,
    device=hub.Device("Samsung Galaxy S24 (Family)"),
    inputs=dict(image=[input_array]),
)


In [ ]:
on_device_output = inference_job.download_output_data()

In [ ]:
# Step 5: Post-processing the on-device output
output_name = list(on_device_output.keys())[0]
out = on_device_output[output_name][0]
on_device_probabilities = np.exp(out) / np.sum(np.exp(out), axis=1)

In [ ]:
# Read the class labels for imagenet
sample_classes = "https://qaihub-public-assets.s3.us-west-2.amazonaws.com/apidoc/imagenet_classes.txt"
response = requests.get(sample_classes, stream=True)
response.raw.decode_content = True
categories = [str(s.strip()) for s in response.raw]

In [ ]:
# Print top five predictions for the on-device model
print("Top-5 On-Device predictions:")
top5_classes = np.argsort(on_device_probabilities[0], axis=0)[-5:]
for c in reversed(top5_classes):
    print(f"{c} {categories[c]:20s} {on_device_probabilities[0][c]:>6.1%}")

# Run inference on Torch model

In [ ]:
torch_model = mobilenet_v2(pretrained=True)
torch_model.eval()

# Convert the input_array (which is a NumPy array) into a torch tensor.
input_tensor = torch.from_numpy(input_array)

# Run inference with the Torch model
with torch.no_grad():
    torch_output = torch_model(input_tensor)

# Convert the output to a NumPy array
torch_output_np = torch_output.cpu().numpy()

# Post-process the Torch model's output with softmax
torch_probabilities = np.exp(torch_output_np) / np.sum(np.exp(torch_output_np), axis=1, keepdims=True)

# Print top five predictions for the Torch model
print("\nTop-5 Torch model predictions:")
top5_indices_torch = np.argsort(torch_probabilities[0])[-5:]
for idx in reversed(top5_indices_torch):
    print(f"{idx} {categories[idx]:20s} {torch_probabilities[0][idx]:>6.1%}")

# Step 6: Download model

In [ ]:
# Step 6: Download model
target_model = compile_job.get_target_model()

target_model.download("mobilenet_v2.tflite")

# Optimized and validated by Qualcomm

In [ ]:
%run -m qai_hub_models.models.mobilenet_v2.export

In [ ]:
%run -m qai_hub_models.models.mobilenet_v2.export -- --target-runtime tflite --device "Samsung Galaxy S24 (Family)"

In [ ]:
%run -m qai_hub_models.models.mobilenet_v2.demo

# Quantization

In [ ]:
%run -m qai_hub_models.models.mobilenet_v2_quantized.demo

In [ ]:
%run -m qai_hub_models.models.mobilenet_v2_quantized.export -- --target-runtime tflite --device "Samsung Galaxy S24 (Family)"